# Demo 1 Setup

This notebook creates the demo objects used in **Demo 1: Unity Catalog 3 Level Namespace, Data Discovery and Data Lineage**.

It creates a personal schema and derived tables from the `samples` catalog to generate data lineage.

In [0]:
# Get current user and create a unique schema name
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
schema_name = f"demo_lineage_{clean_username}"
catalog_name = spark.sql("SELECT current_catalog()").first()[0]

# Create the schema and set it as the current schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}` COMMENT 'Demo schema for Unity Catalog exploration - created for Demo 1'")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

print(f"✓ Using: {catalog_name}.{schema_name}")
print(f"  User: {username}")

In [0]:
%sql
-- Create a customers table derived from samples.tpch.customer
CREATE OR REPLACE TABLE customers
COMMENT 'Customer dimension table derived from TPC-H sample data'
AS
SELECT 
  c_custkey AS customer_id,
  c_name AS customer_name,
  c_address AS address,
  n.n_name AS nation,
  c_phone AS phone,
  c_acctbal AS account_balance,
  c_mktsegment AS market_segment,
  c_comment AS comment
FROM samples.tpch.customer c
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey;

In [0]:
%sql
-- Create an orders table derived from samples.tpch.orders
CREATE OR REPLACE TABLE orders
COMMENT 'Orders fact table derived from TPC-H sample data'
AS
SELECT 
  o_orderkey AS order_id,
  o_custkey AS customer_id,
  o_orderstatus AS order_status,
  o_totalprice AS total_price,
  o_orderdate AS order_date,
  o_orderpriority AS order_priority,
  o_clerk AS clerk,
  o_shippriority AS ship_priority
FROM samples.tpch.orders;

In [0]:
%sql
-- Create a revenue_by_nation view that joins customers and orders (generates column-level lineage)
CREATE OR REPLACE VIEW revenue_by_nation
COMMENT 'Aggregated revenue by nation - demonstrates lineage from customers and orders tables'
AS
SELECT 
  c.nation,
  c.market_segment,
  COUNT(o.order_id) AS total_orders,
  ROUND(SUM(o.total_price), 2) AS total_revenue,
  ROUND(AVG(o.total_price), 2) AS avg_order_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.nation, c.market_segment;

In [0]:
%sql
-- Add column comments to make discovery richer
ALTER TABLE customers ALTER COLUMN customer_id COMMENT 'Unique customer identifier';
ALTER TABLE customers ALTER COLUMN account_balance COMMENT 'Current account balance in USD';
ALTER TABLE customers ALTER COLUMN market_segment COMMENT 'Market segment classification (AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD, MACHINERY)';

ALTER TABLE orders ALTER COLUMN order_id COMMENT 'Unique order identifier';
ALTER TABLE orders ALTER COLUMN customer_id COMMENT 'Foreign key to customers table';
ALTER TABLE orders ALTER COLUMN total_price COMMENT 'Total order value in USD';

In [0]:
%sql
-- Query the view to ensure lineage is captured
SELECT * FROM revenue_by_nation ORDER BY total_revenue DESC LIMIT 5;

In [0]:
print("✓ Setup complete!")
print(f"")
print(f"Created in {catalog_name}.{schema_name}:")
print(f"  • Table: customers (derived from samples.tpch.customer + samples.tpch.nation)")
print(f"  • Table: orders (derived from samples.tpch.orders)")
print(f"  • View:  revenue_by_nation (joins customers + orders)")
print(f"")
print(f"Lineage has been generated across these objects.")